In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Ellipse, Circle
from scipy.ndimage import gaussian_filter, label

# Wood Lichtenberg — Resistive Surface Burning

A different notebook from the acrylic one because the physics is different. The acrylic case is pure electrostatics: buried free charge drives bulk dielectric breakdown, the tree propagates as a nanosecond streamer, and the simulator is a dielectric-breakdown model on top of a Poisson solve.

Wood Lichtenberg (Theory §5) runs a slower, quasi-static loop:

1. An electrolyte-wetted surface is a 2D resistive sheet between two point electrodes at fixed voltage. Current obeys $\nabla\!\cdot\!(\sigma\nabla V) = 0$ — Laplace with a *spatially varying, anisotropic* conductivity.
2. Local Joule heating $p = \sigma|\vec E|^2 = |\vec J|^2/\sigma$ raises the temperature via the heat equation.
3. When a cell crosses the pyrolysis threshold $T_\text{pyr}\sim 300^\circ\text{C}$, its cellulose pyrolyses to amorphous carbon — $\sigma$ jumps by ~10³.
4. Char carries current better → field concentrates at the char tips → heat piles up there → more char. Positive feedback.
5. Anisotropy (grain) and stochastic heterogeneity (density variation, moisture pockets) break any symmetry and seed the branching.

No single blinding streamer — just a slowly spreading carbonisation front that competes for current until the two electrodes are bridged. Timescales are seconds to minutes, not nanoseconds; fractal dimensions ($D_f\sim 1.3$–1.6, Theory §4.5) come out lower than the acrylic case because grain anisotropy partially straightens the branches.

Physical Constants and Wood Properties

Values from Theory §5.5. `sigma_wet`/`sigma_char` is the key ratio driving the whole feedback loop — three decades of conductivity jump at pyrolysis.

In [ ]:
WOOD = {
    "eps_r":       4.0,          # dielectric constant (not used in quasi-static resistive problem)
    "sigma_wet":   0.1,          # S/m   surface conductivity of electrolyte-wetted wood
    "sigma_char":  100.0,        # S/m   carbonised wood  (~10³× jump, §5.3)
    "T_pyr":       573.0,        # K     pyrolysis onset (~300°C)
    "T_ambient":   293.0,        # K     20°C
    "rho_m":       500.0,        # kg/m³ softwood
    "c_p":         1500.0,       # J/(kg·K)
    "k_therm":     0.15,         # W/(m·K)
}

cm_to_m = 1e-2

Board Geometry (2D surface)

A rectangular wood board viewed from above. The third dimension (thickness) doesn't enter the model — the wet layer and resulting char both live on the surface. Grain runs along $\hat x$.

In [ ]:
resolution = 0.1                # cm per cell

x_size = 20                     # cm  (along grain)
y_size = 10                     # cm  (across grain)

nx = int(round(x_size / resolution))
ny = int(round(y_size / resolution))
h  = resolution * cm_to_m       # grid spacing [m]

print(f"Board: {x_size} × {y_size} cm  →  grid {nx} × {ny} = {nx*ny} cells  (h = {h*1e3:.1f} mm)")

Initial Conductivity Field — Anisotropy and Heterogeneity

Two features make wood different from an idealised resistive sheet (Theory §5.2):

- **Grain anisotropy.** Conductivity along the vessels (our $\hat x$) is several times higher than across. We encode this as a diagonal conductivity tensor with $\sigma_x = r_\text{aniso}\,\sigma_y$.
- **Stochastic heterogeneity.** Density, moisture, knots. A correlated log-normal noise field (Gaussian-blurred white noise, exponentiated) is a cheap stand-in for the real microstructure; the correlation length sets the typical 'grain-width' scale of resistive variation.

Both anisotropy and heterogeneity matter for branching. A perfectly uniform, isotropic sheet would heat up as a single expanding disc — no branches, no Lichtenberg figure.

In [ ]:
aniso_ratio  = 3.0              # σ_x / σ_y  (along-grain / across-grain)
noise_sigma  = 3                # correlation length of heterogeneity [cells]
noise_amp    = 0.35             # log-amplitude of σ heterogeneity (±35% ~ 1 stdev)
seed         = 1

rng          = np.random.default_rng(seed)
white        = rng.normal(size=(nx, ny))
correlated   = gaussian_filter(white, sigma=noise_sigma)
correlated  /= correlated.std()                           # unit variance

sigma_base   = WOOD["sigma_wet"] * np.exp(noise_amp * correlated)

sigma_x      = aniso_ratio * sigma_base                   # along grain
sigma_y      = sigma_base                                 # across grain

print(f"Initial σ:  [{sigma_base.min():.3e}, {sigma_base.max():.3e}] S/m")
print(f"Anisotropy: σ_x / σ_y = {aniso_ratio}")
print(f"Contrast (max/min): {sigma_base.max() / sigma_base.min():.2f}×")

Adding Knots

The Gaussian-noise heterogeneity above captures mild density/moisture variations but misses the most conspicuous feature of real wood: **knots**. Electrically they matter because:

- **Higher resistance.** Knots are denser and more lignin-rich; electrolyte soaks in less. $\sigma$ inside a knot is typically 10–50× lower than in straight grain.
- **Compact shape.** A few mm to a few cm across, with a sharp-ish boundary.
- **Current deflection.** A low-σ blob forces current to flow around it — exactly the geometry that pushes burn fingers off-axis and seeds branching points. Real wood-burn figures often have a branch visibly hooking around a knot.

Below we add a handful of random knots to `sigma_base` as compact low-conductivity disks with a tanh-softened edge. Placement rejects positions too close to the electrodes (so the wood's morphology biases the burn, not our knot placement). Set `n_knots = 0` to skip knots entirely and recover the plain noisy-wood model.

**What we don't model.** Real knots also *bend the grain around themselves* — wood fibers curl around a knot rather than running through it. Modelling that correctly means letting the anisotropy tensor rotate spatially: $\sigma$ becomes a general symmetric $2\times 2$ matrix with off-diagonal entries where the grain swirls. The Poisson solver above assumes a *diagonal* tensor ($\sigma_{xy}=0$), so adding grain swirl would require extending the finite-volume stencil to cross-terms. Flag for later — the isotropic-knot model below is a reasonable first approximation.

In [ ]:
# --- Knot parameters ---
n_knots            = 4                  # set to 0 to skip knots entirely
knot_radius_cm     = (0.4, 1.2)         # (min, max) radius of each knot
knot_sigma_factor  = 0.05               # σ inside knot = factor × baseline (20× drop)
knot_softness_cm   = 0.2                # tanh edge width
min_clearance_cm   = 2.0                # minimum distance knot-centre to an electrode

# --- Placement (rejection sampling) ---
rng_k        = np.random.default_rng(seed + 100)
knot_centers = []   # list of (xc_cm, yc_cm, r_cm)
attempts     = 0
while len(knot_centers) < n_knots and attempts < 500:
    attempts += 1
    xc = rng_k.uniform(1.5, x_size - 1.5)
    yc = rng_k.uniform(1.5, y_size - 1.5)
    rk = rng_k.uniform(*knot_radius_cm)

    d_L = np.hypot(xc - elec_L[0] * resolution, yc - elec_L[1] * resolution)
    d_R = np.hypot(xc - elec_R[0] * resolution, yc - elec_R[1] * resolution)
    if d_L < min_clearance_cm or d_R < min_clearance_cm:
        continue
    if any(np.hypot(xc - cx, yc - cy) < (rk + rk2 + 0.4)
           for (cx, cy, rk2) in knot_centers):
        continue
    knot_centers.append((xc, yc, rk))

print(f"Placed {len(knot_centers)} knots  (after {attempts} placement attempts)")

# --- Knot field: multiplicative modifier on σ_base ---
xx       = np.arange(nx) * resolution
yy       = np.arange(ny) * resolution
Xg, Yg   = np.meshgrid(xx, yy, indexing='ij')

knot_field = np.ones((nx, ny))
for (xc, yc, rk) in knot_centers:
    dist         = np.sqrt((Xg - xc) ** 2 + (Yg - yc) ** 2)
    mask_outside = 0.5 * (1.0 + np.tanh((dist - rk) / knot_softness_cm))
    # smooth blend: outside → 1.0, inside → knot_sigma_factor
    knot_field  *= mask_outside + (1.0 - mask_outside) * knot_sigma_factor

# --- Apply to conductivity (both axes of the tensor) ---
sigma_base = sigma_base * knot_field
sigma_x    = aniso_ratio * sigma_base
sigma_y    = sigma_base

print(f"σ_base after knots:  [{sigma_base.min():.3e}, {sigma_base.max():.3e}] S/m")
print(f"  contrast (max / min): {sigma_base.max() / sigma_base.min():.1f}×")

# --- Dedicated knot-map view ---
from matplotlib.patches import Circle

fig, ax = plt.subplots(figsize=(10, 4.8))
im = ax.imshow(sigma_base.T, origin='lower', extent=[0, x_size, 0, y_size],
               cmap='YlOrBr_r')
plt.colorbar(im, ax=ax, label='σ_base (S/m)', fraction=0.04)

for i, (xc, yc, rk) in enumerate(knot_centers):
    ax.add_patch(Circle((xc, yc), rk, fill=False,
                        edgecolor='crimson', lw=1.4, linestyle='--'))
    ax.text(xc, yc, f'k{i+1}', color='crimson', ha='center', va='center',
            fontsize=9, fontweight='bold')

ax.plot(elec_L[0] * resolution, elec_L[1] * resolution, marker='^',
        color='royalblue', markersize=10, markeredgecolor='white',
        linestyle='', label='electrodes')
ax.plot(elec_R[0] * resolution, elec_R[1] * resolution, marker='^',
        color='royalblue', markersize=10, markeredgecolor='white',
        linestyle='')

ax.set(xlabel='x (cm, along grain)', ylabel='y (cm, across grain)',
       title=f'Wood model: noise + {len(knot_centers)} knots (dashed circles)',
       aspect='equal')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

Visualising the Wood Model

Three views of the input field before the burn starts:

1. **Heterogeneity map $\sigma_\text{base}$.** The Gaussian-filtered log-normal noise. Blobs of slightly higher or lower conductivity stand in for density bands, moisture pockets, and knots — the real microstructural disorder that seeds branch nucleation. Correlation length = `noise_sigma` × `h` ≈ 3 mm at default settings.

2. **Anisotropic tensor.** The actual conductivity at each point is a diagonal tensor $\sigma(\vec r) = \operatorname{diag}(\sigma_x, \sigma_y) = \operatorname{diag}(r_\text{aniso}\,\sigma_\text{base},\ \sigma_\text{base})$. Ellipses in the middle panel visualise it: major axis along grain ($\hat x$), minor axis across. Heterogeneity modulates the *overall magnitude* of the tensor without changing its axis orientation — at every point, current wants to flow along $\hat x$ by the same factor $r_\text{aniso}$.

3. **Directional response.** Fed a uniform $\vec E$ at 45° to the grain, the current density $\vec J = \sigma\vec E$ tilts *toward* $\hat x$ because $\sigma_x > \sigma_y$. An isotropic sheet would have $\vec J \parallel \vec E$; the bend angle $45° - \arctan(1/r_\text{aniso})$ is the observable signature of anisotropy and is exactly what makes burn fingers prefer to run with the grain.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# === Panel 1: σ_base heterogeneity ===
ax = axes[0]
im = ax.imshow(sigma_base.T, origin='lower', extent=[0, x_size, 0, y_size],
               cmap='YlOrBr_r')
plt.colorbar(im, ax=ax, label='σ_base (S/m)', fraction=0.04)
ax.set(xlabel='x (cm, along grain)', ylabel='y (cm, across grain)',
       title='1. σ_base — heterogeneity\n(Gaussian-filtered log-normal noise)',
       aspect='equal')

# === Panel 2: conductivity-tensor ellipses over σ_base ===
ax = axes[1]
im = ax.imshow(sigma_base.T, origin='lower', extent=[0, x_size, 0, y_size],
               cmap='YlOrBr_r', alpha=0.6)
plt.colorbar(im, ax=ax, label='σ_base (S/m)', fraction=0.04)

n_ex, n_ey = 7, 4
xs_e = np.linspace(x_size / (n_ex + 1), x_size * n_ex / (n_ex + 1), n_ex)
ys_e = np.linspace(y_size / (n_ey + 1), y_size * n_ey / (n_ey + 1), n_ey)
ell_major = 0.75 * (x_size / n_ex)           # along grain
ell_minor = ell_major / aniso_ratio          # across grain
for xe in xs_e:
    for ye in ys_e:
        ax.add_patch(Ellipse((xe, ye), width=ell_major, height=ell_minor,
                             facecolor='none', edgecolor='midnightblue',
                             lw=1.6, alpha=0.95))
ax.set(xlabel='x (cm)', ylabel='y (cm)', aspect='equal',
       title=f'2. σ tensor  —  σ_x / σ_y = {aniso_ratio}\n'
             f'(ellipse axes ∝ σ along each direction)')

# === Panel 3: response to uniform E at 45° to grain ===
ax = axes[2]

E0        = 1.0
Ex_test   = E0 / np.sqrt(2)
Ey_test   = E0 / np.sqrt(2)

Jx_field  = sigma_x * Ex_test
Jy_field  = sigma_y * Ey_test
J_mag_f   = np.sqrt(Jx_field ** 2 + Jy_field ** 2)

im = ax.imshow(J_mag_f.T, origin='lower', extent=[0, x_size, 0, y_size],
               cmap='magma')
plt.colorbar(im, ax=ax, label='|J| (A/m² at E = 1 V/m)', fraction=0.04)

# Quiver of J direction at a sparse grid
n_qx, n_qy = 14, 7
xs_q = np.linspace(x_size / (n_qx + 1), x_size * n_qx / (n_qx + 1), n_qx)
ys_q = np.linspace(y_size / (n_qy + 1), y_size * n_qy / (n_qy + 1), n_qy)
Xq, Yq = np.meshgrid(xs_q, ys_q, indexing='xy')
ixq = np.clip((Xq / resolution).astype(int), 0, nx - 1)
iyq = np.clip((Yq / resolution).astype(int), 0, ny - 1)
Jxq = sigma_x[ixq, iyq] * Ex_test
Jyq = sigma_y[ixq, iyq] * Ey_test
ax.quiver(Xq, Yq, Jxq, Jyq, color='cyan', width=0.005,
          scale_units='xy', scale=(Jxq.max()) / 0.9, alpha=0.95)

# Reference arrow: the applied E direction
ax.annotate('', xy=(2.0, 2.0), xytext=(0.6, 0.6),
            arrowprops=dict(arrowstyle='->', color='white', lw=2.2,
                            linestyle='--'))
ax.text(0.8, 2.3, 'applied E  (45°)', color='white', fontsize=9,
        fontweight='bold')

deflect = 45.0 - np.degrees(np.arctan(1.0 / aniso_ratio))
ax.set(xlabel='x (cm)', ylabel='y (cm)', aspect='equal',
       title=f'3. J under uniform E at 45°\n'
             f'deflected {deflect:.1f}° toward grain axis')

plt.tight_layout()
plt.show()

print(f"Isotropic expectation:  J would be at 45° (parallel to E).")
print(f"Anisotropic result:     J at arctan(σ_y/σ_x) = "
      f"arctan(1/{aniso_ratio:.1f}) = "
      f"{np.degrees(np.arctan(1/aniso_ratio)):.1f}°")
print(f"Deflection toward grain: {deflect:.1f}°")
print(f"(A burn finger sampling any random direction feels this bias on every step.)")

Electrodes

Two point electrodes driven into the board — one at the applied voltage, one grounded. Placing them along the grain line keeps the initial current axis horizontal so branching off-axis is easy to see. $V_\text{applied}\sim\text{kV}$ is typical of a DIY microwave-oven transformer (MOT) setup after rectification.

In [ ]:
V_applied = 5000.0                              # V  (peak DC-equivalent of a MOT + bridge rectifier)

elec_L = (int(0.10 * nx), ny // 2)              # left electrode (high voltage)
elec_R = (int(0.90 * nx), ny // 2)              # right electrode (ground)

is_electrode = np.zeros((nx, ny), dtype=bool)
is_electrode[elec_L] = True
is_electrode[elec_R] = True

electrode_V  = np.zeros((nx, ny))
electrode_V[elec_L] = V_applied
electrode_V[elec_R] = 0.0

print(f"V applied: {V_applied:.0f} V  across electrodes")
print(f"Electrodes at: {elec_L} and {elec_R}")
print(f"Spacing: {(elec_R[0]-elec_L[0])*resolution:.1f} cm")

Variable-Conductivity Poisson Solve

Current conservation in the steady state gives

$$\nabla\!\cdot\!\vec J = 0,\qquad \vec J = -\sigma\nabla V,$$

so with $\sigma$ spatially varying and anisotropic,

$$\partial_x(\sigma_x\,\partial_x V) + \partial_y(\sigma_y\,\partial_y V) = 0,$$

subject to $V$ fixed at the two electrodes and $\partial_n V = 0$ on the four free board edges (no current leaks out of the sides).

Because $\sigma$ is not constant — and by the end of the run it will vary over three decades — the FFT trick from the acrylic notebook doesn't apply. We use a finite-volume discretisation (conductivities are arithmetically averaged onto the cell faces) and **SOR** iteration. Warm-starting from the previous timestep's $V$ keeps per-step sweep counts low once the solution has settled.

In [ ]:
def solve_potential(V, sigma_x, sigma_y, electrode_V, is_electrode,
                    n_iters=40, omega=1.85):
    """SOR sweep on ∂_x(σ_x ∂_x V) + ∂_y(σ_y ∂_y V) = 0.

    Dirichlet at electrode cells, Neumann (copy-neighbour) at board edges.
    Face conductivities are arithmetic means of adjacent cell values.
    """
    V[is_electrode] = electrode_V[is_electrode]

    for _ in range(n_iters):
        # Face-centred conductivities (arithmetic mean of the two cells they separate)
        sxp = 0.5 * (sigma_x[2:,   1:-1] + sigma_x[1:-1, 1:-1])   # σ_x at i+½
        sxm = 0.5 * (sigma_x[:-2,  1:-1] + sigma_x[1:-1, 1:-1])   # σ_x at i-½
        syp = 0.5 * (sigma_y[1:-1, 2:  ] + sigma_y[1:-1, 1:-1])   # σ_y at j+½
        sym = 0.5 * (sigma_y[1:-1, :-2 ] + sigma_y[1:-1, 1:-1])   # σ_y at j-½

        denom = sxp + sxm + syp + sym
        V_gs  = (sxp * V[2:,   1:-1] + sxm * V[:-2,  1:-1] +
                 syp * V[1:-1, 2:  ] + sym * V[1:-1, :-2 ]) / denom

        interior = (slice(1, -1), slice(1, -1))
        V[interior] = V[interior] + omega * (V_gs - V[interior])

        # Neumann at board edges: copy inward neighbour
        V[0,  :] = V[1,  :]
        V[-1, :] = V[-2, :]
        V[:,  0] = V[:,  1]
        V[:, -1] = V[:, -2]

        # Dirichlet at electrodes
        V[is_electrode] = electrode_V[is_electrode]

    return V

# Initial solve to warm-start the simulation
V = np.full((nx, ny), 0.5 * V_applied)
V = solve_potential(V, sigma_x, sigma_y, electrode_V, is_electrode, n_iters=400)

dVdx = np.gradient(V, h, axis=0)
dVdy = np.gradient(V, h, axis=1)
Jx   = -sigma_x * dVdx
Jy   = -sigma_y * dVdy
J_mag = np.sqrt(Jx**2 + Jy**2)

print(f"V range: [{V.min():.1f}, {V.max():.1f}] V")
print(f"|J| range: [{J_mag.min():.3e}, {J_mag.max():.3e}] A/m²")
print(f"Current density peaks at the electrodes (geometric singularity).")

Heat Equation + Pyrolysis Feedback

With the conductivity field fixed, the current distribution is known and deposits power at density $p = \sigma_x(\partial_x V)^2 + \sigma_y(\partial_y V)^2$. The temperature evolves via

$$\rho_m c_p\,\partial_t T = p + k\,\nabla^2 T,$$

discretised with explicit Euler in time and a 5-point Laplacian in space. Stability requires $\Delta t < h^2/(4\alpha)$ where $\alpha = k/(\rho_m c_p)\approx 2\times 10^{-7}\,\text{m}^2/\text{s}$; at $h=1$ mm that gives $\Delta t < 1.25$ s, so $\Delta t = 0.05$ s is comfortable.

**Pyrolysis switch.** When a cell's $T$ first exceeds `T_pyr`, its conductivity is promoted to `sigma_char` (isotropic — the cellular structure has disintegrated into amorphous carbon). That cell is now a low-resistance conductor; current redistributes on the next `solve_potential` call to concentrate at *its* tips. The loop runs until the electrodes are bridged by char (visible as a current spike) or the step budget is exhausted.

In [ ]:
# --- Simulation parameters ---
dt             = 0.05            # s
n_steps        = 1200            # 60 s of simulated burning
poisson_iters  = 25              # per-step SOR sweeps (warm-started)

save_frames    = 8
save_stride    = max(1, n_steps // save_frames)

# --- State ---
T            = np.full((nx, ny), WOOD["T_ambient"])
carbonised   = np.zeros((nx, ny), dtype=bool)
carb_step    = np.full((nx, ny), -1, dtype=np.int32)

snapshots    = {"step": [], "t": [], "T": [], "char": [], "J": []}

alpha_thermal = WOOD["k_therm"] / (WOOD["rho_m"] * WOOD["c_p"])
dt_stab       = h**2 / (4 * alpha_thermal)
print(f"α_thermal = {alpha_thermal:.3e} m²/s")
print(f"Thermal stability limit: dt < {dt_stab:.2f} s  (using dt = {dt} s)")
print()

bridged       = False
bridge_step   = None

for step in range(1, n_steps + 1):
    # 1. Current distribution for the current σ field (warm-start from prev V)
    V = solve_potential(V, sigma_x, sigma_y, electrode_V, is_electrode,
                        n_iters=poisson_iters, omega=1.85)

    # 2. Gradients + current density
    dVdx = np.gradient(V, h, axis=0)
    dVdy = np.gradient(V, h, axis=1)
    Jx   = -sigma_x * dVdx
    Jy   = -sigma_y * dVdy

    # 3. Volumetric Joule power  p = J·E = σ_x Ex² + σ_y Ey²
    p    = sigma_x * dVdx**2 + sigma_y * dVdy**2

    # 4. Heat equation (explicit Euler + 5-point Laplacian)
    lap_T = np.zeros_like(T)
    lap_T[1:-1, 1:-1] = (T[2:, 1:-1] + T[:-2, 1:-1] +
                         T[1:-1, 2:] + T[1:-1, :-2] - 4.0 * T[1:-1, 1:-1]) / h**2
    T += dt * (p + WOOD["k_therm"] * lap_T) / (WOOD["rho_m"] * WOOD["c_p"])

    # Electrodes stay at ambient (they're metal, they dump heat)
    T[is_electrode] = WOOD["T_ambient"]

    # 5. Pyrolysis: freshly-crossed cells become char
    newly_char = (T > WOOD["T_pyr"]) & (~carbonised) & (~is_electrode)
    if newly_char.any():
        sigma_x[newly_char]   = WOOD["sigma_char"]
        sigma_y[newly_char]   = WOOD["sigma_char"]
        carbonised           |= newly_char
        carb_step[newly_char] = step

    # 6. Check for electrode bridging (char-connected)
    if not bridged and carbonised.any():
        from scipy.ndimage import label
        reachable_from_L = carbonised.copy()
        reachable_from_L[elec_L] = True
        labelled, _ = label(reachable_from_L)
        L_comp = labelled[elec_L]
        if L_comp != 0 and L_comp == labelled[elec_R] and carbonised[elec_R]:
            # unlikely direct-cell bridging; use a more lenient criterion:
            pass
        # lenient: char cell within 2 cells of each electrode lies in the same component
        labelled_all, _ = label(carbonised)
        near_L_labels = set(labelled_all[max(0,elec_L[0]-2):elec_L[0]+3,
                                          max(0,elec_L[1]-2):elec_L[1]+3].ravel()) - {0}
        near_R_labels = set(labelled_all[max(0,elec_R[0]-2):elec_R[0]+3,
                                          max(0,elec_R[1]-2):elec_R[1]+3].ravel()) - {0}
        if near_L_labels & near_R_labels:
            bridged     = True
            bridge_step = step
            print(f"step {step:4d} (t={step*dt:5.2f}s): char BRIDGES the electrodes.")

    # 7. Save snapshot
    if step % save_stride == 0 or step == 1 or step == n_steps:
        J_mag = np.sqrt(Jx**2 + Jy**2)
        snapshots["step"].append(step)
        snapshots["t"].append(step * dt)
        snapshots["T"].append(T.copy())
        snapshots["char"].append(carbonised.copy())
        snapshots["J"].append(J_mag)
        n_char = int(carbonised.sum())
        print(f"step {step:4d} (t={step*dt:5.2f}s) | T_max = {T.max():6.1f} K | "
              f"char: {n_char:5d} cells ({n_char/(nx*ny)*100:4.1f}%) | "
              f"|J|_max = {J_mag.max():.2e} A/m²")

    # End early if current has spiked very high (post-bridging, simulation becomes unphysical)
    if bridged and step > bridge_step + 50:
        print(f"step {step}: 50 steps past bridging, stopping.")
        break

print(f"\nDone. Carbonised cells: {int(carbonised.sum())} / {nx*ny} "
      f"({carbonised.sum()/(nx*ny)*100:.1f}%).")
if bridge_step is not None:
    print(f"Electrodes bridged at step {bridge_step} (t={bridge_step*dt:.2f} s).")

Burn Progression

Snapshots of the carbonised region, temperature, and current density at evenly-spaced moments during the run. Watch the fingers advance from each electrode, bifurcate at heterogeneity, and eventually join — the moment they join (bridging), the current surges and in a real setup the fuses blow / the MOT buzzes alarmingly.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

n_frames = len(snapshots["step"])
ncols    = min(4, n_frames)
nrows    = int(np.ceil(n_frames / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 2.2*nrows), squeeze=False)
for idx, ax in enumerate(axes.flat):
    if idx >= n_frames:
        ax.axis('off')
        continue
    char = snapshots["char"][idx]
    T_s  = snapshots["T"][idx]
    ax.imshow(T_s.T, origin='lower', extent=[0, x_size, 0, y_size],
              cmap='inferno', vmin=WOOD["T_ambient"], vmax=max(WOOD["T_pyr"]*1.2, T_s.max()))
    # Overlay char as contour
    if char.any():
        ax.contour(char.T.astype(float), levels=[0.5],
                   extent=[0, x_size, 0, y_size],
                   colors='cyan', linewidths=0.6)
    ax.plot(elec_L[0]*resolution, elec_L[1]*resolution, 'wo', markersize=6,
            markeredgecolor='black')
    ax.plot(elec_R[0]*resolution, elec_R[1]*resolution, 'wo', markersize=6,
            markeredgecolor='black')
    ax.set_title(f't = {snapshots["t"][idx]:.1f} s | '
                 f'{int(char.sum())} char cells', fontsize=9)
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')

fig.suptitle('Temperature field (inferno) + carbonised region (cyan contour)', fontsize=11)
plt.tight_layout()
plt.show()

Final Current Density and Char Pattern

$|J|$ is what actually deposits energy; in the final state it concentrates hugely along the char network because $\sigma$ is 10³× higher there and the voltage-drop path of least resistance runs through the char. Log-scale color because the dynamic range of $|J|$ between quiet uncharred wood and the carbonised channels spans several decades.

In [ ]:
J_mag_final = snapshots["J"][-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Final char pattern
ax = axes[0]
bg = ax.imshow(sigma_base.T, origin='lower', extent=[0, x_size, 0, y_size],
               cmap='Greys', alpha=0.3)
char_rgba = np.zeros(carbonised.T.shape + (4,))
char_rgba[carbonised.T] = [0.1, 0.05, 0.0, 1.0]    # dark brown for char
ax.imshow(char_rgba, origin='lower', extent=[0, x_size, 0, y_size])
ax.plot(elec_L[0]*resolution, elec_L[1]*resolution, 'ro', markersize=8, label='V+ electrode')
ax.plot(elec_R[0]*resolution, elec_R[1]*resolution, 'bo', markersize=8, label='GND electrode')
ax.set(xlabel='x (cm)', ylabel='y (cm)',
       title=f'Final carbonised pattern  ({int(carbonised.sum())} cells)')
ax.legend(loc='upper right')

# Final current density
ax = axes[1]
J_plot = np.maximum(J_mag_final, 1e-3)
im = ax.imshow(J_plot.T, origin='lower', extent=[0, x_size, 0, y_size],
               cmap='inferno', norm=LogNorm(vmin=J_plot.min(), vmax=J_plot.max()))
ax.plot(elec_L[0]*resolution, elec_L[1]*resolution, 'wo', markersize=6, markeredgecolor='cyan')
ax.plot(elec_R[0]*resolution, elec_R[1]*resolution, 'wo', markersize=6, markeredgecolor='cyan')
plt.colorbar(im, ax=ax, label='|J|  (A/m², log)')
ax.set(xlabel='x (cm)', ylabel='y (cm)', title='Final |J| distribution')

plt.tight_layout()
plt.show()

Fractal Analysis

Box-count the final carbonised pattern the same way we did in the acrylic notebook (Theory §1.1). Expected range per Theory §4.5: $D_f\approx 1.3$–1.6 for wood. The value will depend on the anisotropy ratio (higher anisotropy → more grain-following branches → lower $D_f$) and on whether the run terminated cleanly or bridged early.

In [ ]:
def box_count(binary, box_sizes):
    counts = []
    H, W = binary.shape
    for s in box_sizes:
        Hs, Ws = (H // s) * s, (W // s) * s
        block  = binary[:Hs, :Ws].reshape(Hs // s, s, Ws // s, s)
        counts.append(int(block.any(axis=(1, 3)).sum()))
    return np.array(counts)

def lacunarity(binary, s):
    H, W = binary.shape
    if s >= min(H, W):
        return np.nan
    ii = np.pad(np.cumsum(np.cumsum(binary.astype(np.int64), 0), 1),
                ((1, 0), (1, 0)))
    mass = (ii[s:, s:] - ii[:-s, s:] - ii[s:, :-s] + ii[:-s, :-s]).astype(float)
    m1   = mass.mean()
    return (mass ** 2).mean() / (m1 ** 2) if m1 > 0 else np.nan

max_box   = max(1, min(carbonised.shape) // 3)
box_sizes = np.unique(np.round(np.logspace(0, np.log10(max_box), 10)).astype(int))
counts    = box_count(carbonised, box_sizes)

mask              = counts > 0
log_s             = np.log(box_sizes[mask])
log_N             = np.log(counts[mask])
slope, intercept  = np.polyfit(log_s, log_N, 1)
D_f               = -slope

lac = np.array([lacunarity(carbonised, s) for s in box_sizes])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(carbonised.T, origin='lower', extent=[0, x_size, 0, y_size],
               cmap='inferno', interpolation='nearest')
axes[0].plot(elec_L[0]*resolution, elec_L[1]*resolution, 'co', markersize=6)
axes[0].plot(elec_R[0]*resolution, elec_R[1]*resolution, 'co', markersize=6)
axes[0].set(xlabel='x (cm)', ylabel='y (cm)',
            title='Carbonised pattern  (2D)')

axes[1].loglog(box_sizes[mask], counts[mask], 'o', color='steelblue', label='N(ε)')
fit_N = np.exp(intercept) * box_sizes[mask] ** slope
axes[1].loglog(box_sizes[mask], fit_N, '--', color='black',
               label=f'$D_f$ = {D_f:.3f}')
axes[1].set(xlabel='box size ε (px)', ylabel='N(ε)',
            title='Box-counting dimension')
axes[1].legend()

lac_mask = ~np.isnan(lac)
axes[2].loglog(box_sizes[lac_mask], lac[lac_mask], 's-', color='darkred')
axes[2].set(xlabel='box size ε (px)', ylabel='Λ(ε)',
            title='Lacunarity')

plt.tight_layout()
plt.show()

print(f"Box-counting fractal dimension:  D_f = {D_f:.3f}")
print(f"Expected range for wood Lichtenberg (Theory §4.5):  1.3 – 1.6")